# Inclusive Speaker Verification — Interactive Demo
**Team:** Sachin Patil · Venkatesh Doddi · Akash Badola · Ashish Gajbhiye

This notebook walks through the full pipeline on synthetic data.
No datasets need to be downloaded to run it.

In [ ]:
import sys
sys.path.insert(0, '..')
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.features import FeatureExtractor
from src.model import XVectorModel, ECAPAModel
from src.augment import GaussianNoiseAugmenter
from src.evaluate import compute_eer, Evaluator
from src.fairness import FairnessAnalyzer
from src.explain import SaliencyExplainer
from src.utils import set_seed, get_device

set_seed(42)
device = get_device()
print(f'Using device: {device}')

## 1. Feature Extraction

In [ ]:
extractor = FeatureExtractor(feature_type='fbank', n_mels=80)

# Simulate a 2-second utterance
dummy_audio = torch.randn(32000) * 0.01
features    = extractor(dummy_audio.unsqueeze(0))
print(f'Feature shape: {features.shape}  (batch, time_frames, mel_bins)')

plt.figure(figsize=(12, 3))
plt.imshow(features[0].T.numpy(), aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(label='log-Mel energy')
plt.title('Log-Mel Filterbank Features')
plt.xlabel('Frame')
plt.ylabel('Mel bin')
plt.tight_layout()
plt.show()

## 2. Speaker Embeddings (X-Vector & ECAPA-TDNN)

In [ ]:
xvector = XVectorModel(input_dim=80, embedding_dim=512, num_speakers=50)
ecapa   = ECAPAModel(input_dim=80, embedding_dim=192, num_speakers=50)

emb_xv = xvector.get_embedding(features)
emb_ec = ecapa.get_embedding(features)

print(f'X-Vector embedding shape : {emb_xv.shape}')
print(f'ECAPA embedding shape    : {emb_ec.shape}')

# Compare two utterances from same speaker (should score high)
utt1 = extractor(torch.randn(32000).unsqueeze(0) * 0.01)
utt2 = extractor(torch.randn(32000).unsqueeze(0) * 0.01)

import torch.nn.functional as F
e1 = F.normalize(xvector.get_embedding(utt1), dim=-1)
e2 = F.normalize(xvector.get_embedding(utt2), dim=-1)
score = (e1 * e2).sum().item()
print(f'\nCosine similarity (random embeddings): {score:.4f}')

## 3. EER Evaluation

In [ ]:
np.random.seed(42)

# Simulate well-separated scores for demonstration
genuine_scores  = np.random.normal(0.70, 0.08, 300)
impostor_scores = np.random.normal(0.25, 0.10, 300)

scores = np.concatenate([genuine_scores, impostor_scores])
labels = np.array([1]*300 + [0]*300)

eer, threshold = compute_eer(scores, labels)
print(f'EER: {eer*100:.2f}%  (threshold: {threshold:.3f})')

plt.figure(figsize=(8, 4))
plt.hist(genuine_scores,  bins=40, alpha=0.6, label='Genuine',  color='#1B6CA8')
plt.hist(impostor_scores, bins=40, alpha=0.6, label='Impostor', color='#E84545')
plt.axvline(threshold, color='black', linestyle='--', linewidth=1.5, label=f'Threshold @ EER')
plt.title(f'Score Distribution  (EER = {eer*100:.1f}%)')
plt.xlabel('Cosine similarity')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Fairness Analysis

In [ ]:
import os
os.makedirs('results/fairness', exist_ok=True)

# Simulate demographic bias: female speakers have worse EER
n = 200
male_genuine    = np.random.normal(0.72, 0.07, n//2)
male_impostor   = np.random.normal(0.25, 0.09, n//2)
female_genuine  = np.random.normal(0.60, 0.09, n//2)  # lower scores = higher EER
female_impostor = np.random.normal(0.28, 0.10, n//2)

all_scores = np.concatenate([male_genuine, male_impostor, female_genuine, female_impostor])
all_labels = np.array([1]*(n//2) + [0]*(n//2) + [1]*(n//2) + [0]*(n//2))
all_metas  = ([{'gender1': 'male',   'nationality1': 'native'}]     * n +
              [{'gender1': 'female', 'nationality1': 'non_native'}]  * n)

fa = FairnessAnalyzer(output_dir='results/fairness')
results = fa.per_group_metrics(all_scores, all_labels, all_metas, group_keys=['gender1'])

print(f"Overall EER : {results['overall']['eer_pct']:.2f}%")
for group, metrics in results.get('gender1', {}).items():
    print(f"  {group:12s}: EER={metrics['eer_pct']:.2f}%  "
          f"95%CI=[{metrics['ci_low']:.2f}, {metrics['ci_high']:.2f}]")
gap = results.get('fairness_gap', {}).get('gender1', 0)
print(f"\nFairness gap (gender): {gap:.2f}%")

## 5. Threshold Calibration (Mitigation)

In [ ]:
thresholds = fa.calibrate_thresholds(all_scores, all_labels, all_metas,
                                      group_key='gender1', target_fpr=0.05)
print('Calibrated thresholds:', thresholds)

# Apply calibrated thresholds
mitigated = all_scores.copy()
for i, meta in enumerate(all_metas):
    g = meta.get('gender1', 'unknown')
    if g in thresholds:
        mitigated[i] = all_scores[i] - thresholds[g] * 0.05

results_after = fa.per_group_metrics(mitigated, all_labels, all_metas, group_keys=['gender1'])

print('\n--- After mitigation ---')
for group, metrics in results_after.get('gender1', {}).items():
    print(f"  {group:12s}: EER={metrics['eer_pct']:.2f}%")
gap_after = results_after.get('fairness_gap', {}).get('gender1', 0)
print(f"\nFairness gap after: {gap_after:.2f}% (was {gap:.2f}%)")

fa.plot_eer_comparison(results, results_after, group_key='gender1',
                       filename='eer_comparison_notebook.png')
print('Plot saved to results/fairness/eer_comparison_notebook.png')

## 6. Saliency Explanation

In [ ]:
import os
os.makedirs('results/explainability', exist_ok=True)

model_for_explain = XVectorModel(input_dim=80, embedding_dim=512, num_speakers=50)
explainer = SaliencyExplainer(
    model=model_for_explain,
    feature_extractor=extractor,
    device=torch.device('cpu'),
    output_dir='results/explainability',
    n_steps=20,
)

wav1 = torch.randn(32000) * 0.05
wav2 = torch.randn(32000) * 0.05
attr1, attr2 = explainer.explain_pair(wav1, wav2, label=1, utterance_id='demo')

print(f'Attribution shape: {attr1.shape}  (time_frames, mel_bins)')
print(f'Max attribution:   {np.abs(attr1).max():.4f}')
print(f'Saved to: results/explainability/saliency_demo.png')

## Summary

In [ ]:
print('=' * 55)
print('  INCLUSIVE SPEAKER VERIFICATION — SUMMARY')
print('=' * 55)
print(f'  Overall EER          : {results["overall"]["eer_pct"]:.2f}%')
print(f'  Fairness gap (before): {gap:.2f}%')
print(f'  Fairness gap (after) : {gap_after:.2f}%')
print(f'  Improvement          : {gap - gap_after:+.2f}%')
print('=' * 55)